In [8]:

import pandas as pd
import glob

files = sorted(glob.glob("data/**/financials-annual*.csv.gz", recursive=True))
print("Found:", files)

bq = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(bq.shape)
print(bq.columns.tolist())

Found: ['data/financials-annual_0_0_0.csv.gz', 'data/financials-annual_1_0_0.csv.gz']
(68109, 65)
['BQ_ACCOUNTS_PAYABLE', 'BQ_ASSET_TURNOVER', 'BQ_BUILDINGS_AND_OTHER_DEPRECIABLE_ASSETS', 'BQ_CASH', 'BQ_COMPANY_NAME', 'BQ_COR', 'BQ_COR_PER_EMP', 'BQ_CURRENT_ASSETS', 'BQ_CURRENT_LIABILITIES', 'BQ_CURRENT_RATIO', 'BQ_DEBT_TO_EQUITY_RATIO', 'BQ_DEPLETABLE_ASSETS', 'BQ_EBITDA', 'BQ_EBITDA_MARGIN', 'BQ_EBITDA_PER_EMP', 'BQ_FISCAL_YEAR_END', 'BQ_GROSS_PROFIT', 'BQ_GROSS_PROFIT_MARGIN', 'BQ_GROSS_PROFIT_PER_EMP', 'BQ_ID', 'BQ_INTANGIBLE_ASSETS_AMORTIZABLE', 'BQ_INVENTORIES', 'BQ_LAND', 'BQ_LESS_ACCUMULATED_AMORTIZATION', 'BQ_LESS_ACCUMULATED_DEPLETION', 'BQ_LESS_ACCUMULATED_DEPRECIATION', 'BQ_LESS_ALLOWANCE_FOR_BAD_DEBTS', 'BQ_LOANS_FROM_SHAREHOLDERS', 'BQ_LOANS_TO_SHAREHOLDERS', 'BQ_LONG_TERM_DEBT', 'BQ_MORTGAGE_AND_REAL_ESTATE_LOANS', 'BQ_NET_INCOME', 'BQ_NET_INCOME_PER_EMP', 'BQ_NET_PROFIT_MARGIN', 'BQ_NON_CURRENT_ASSETS', 'BQ_NON_CURRENT_LIABILITIES', 'BQ_OPERATING_EXPENSES', 'BQ_OPERATIN

In [9]:
import numpy as np

d = bq.copy()
d.columns = [c.lower().replace("bq_", "") for c in d.columns]
cols = ["ebitda", "operating_income", "buildings_and_other_depreciable_assets", "depletable_assets",
        "land", "less_accumulated_depreciation", "less_accumulated_depletion",
        "current_assets", "cash", "current_liabilities", "short_term_debt"]
for c in cols:
    d[c] = pd.to_numeric(d[c], errors="coerce")
d = d.sort_values(["id", "year"]).drop_duplicates(["id", "year"])

# Derived items
d["da"] = d.ebitda - d.operating_income
d["net_ppe"] = (d[["buildings_and_other_depreciable_assets", "depletable_assets", "land"]].fillna(0).sum(axis=1)
                - d.less_accumulated_depreciation.abs().fillna(0) - d.less_accumulated_depletion.abs().fillna(0))
d["nwc"] = (d.current_assets - d.cash.fillna(0)) - (d.current_liabilities - d.short_term_debt.fillna(0))

# Year-over-year changes, only when the previous row is exactly the prior year
prev = d.groupby("id")[["year", "net_ppe", "nwc"]].shift(1)
ok = prev.year == d.year - 1
d["capex"] = np.where(ok, d.net_ppe - prev.net_ppe + d.da, np.nan)
d["d_nwc"] = np.where(ok, d.nwc - prev.nwc, np.nan)

pos = d[d.ebitda > 0].copy()
pos["da_pct"] = pos.da / pos.ebitda
pos["capex_nwc_pct"] = (pos.capex + pos.d_nwc) / pos.ebitda

# One typical value per company (median across its years), to avoid year-to-year noise
per_co = pos.groupby("id")[["da_pct", "capex_nwc_pct"]].median()
per_co = per_co[per_co.da_pct.between(0, 0.8) & per_co.capex_nwc_pct.between(-0.5, 1.5)]

# EBITDA growth: 5-year CAGR, positive EBITDA at both ends
HOLD = 5
e = d[["id", "year", "ebitda"]].dropna()
fwd = e.assign(year=e.year - HOLD)
pairs = e.merge(fwd, on=["id", "year"], suffixes=("", "_fwd"))
pairs = pairs[(pairs.ebitda > 0) & (pairs.ebitda_fwd > 0)]
growth = ((pairs.ebitda_fwd / pairs.ebitda) ** (1 / HOLD) - 1)
growth = growth[growth.between(-0.30, 0.50)]

summary = {"ebitda_growth": growth, "da_pct": per_co.da_pct, "capex_nwc_pct": per_co.capex_nwc_pct}
stats = pd.DataFrame({k: {"n": len(s), "p10": s.quantile(.1), "p50": s.quantile(.5), "p90": s.quantile(.9)}
                      for k, s in summary.items()}).T
print(stats.round(3))

import os
os.makedirs("results", exist_ok=True)
stats.to_csv("results/brightquery_ranges.csv")

                     n    p10    p50    p90
ebitda_growth  39848.0 -0.063  0.053  0.242
da_pct           810.0  0.102  0.364  0.698
capex_nwc_pct    810.0 -0.008  0.431  0.963


In [10]:
d["revenue"] = pd.to_numeric(d["revenue"], errors="coerce")
d["margin"] = d.ebitda / d.revenue

screen = d.groupby("id").agg(
    n_years=("year", "count"),
    always_pos=("ebitda", lambda s: (s > 0).all()),
    margin=("margin", "median"),
)
keep = screen[(screen.n_years >= 5) & screen.always_pos & (screen.margin >= 0.10)].index
co_pe = co[co.index.isin(keep)]
co_pe.to_csv("data/brightquery_company_params_pe.csv")

print(f"All companies: {len(co)}   PE-style screened: {len(co_pe)}")
print("\nAll companies:\n", co.describe().round(3))
print("\nScreened:\n", co_pe.describe().round(3))

All companies: 773   PE-style screened: 562

All companies:
             da  capex_nwc  ebitda_g
count  773.000    773.000   773.000
mean     0.387      0.456     0.055
std      0.216      0.385     0.087
min      0.001     -0.494    -0.230
25%      0.201      0.196     0.006
50%      0.374      0.444     0.043
75%      0.565      0.694     0.088
max      0.795      1.500     0.411

Screened:
             da  capex_nwc  ebitda_g
count  562.000    562.000   562.000
mean     0.383      0.445     0.050
std      0.211      0.369     0.084
min      0.001     -0.484    -0.230
25%      0.200      0.195     0.007
50%      0.368      0.423     0.042
75%      0.547      0.676     0.080
max      0.795      1.500     0.408


In [11]:
pairs["g"] = (pairs.ebitda_fwd / pairs.ebitda) ** (1 / HOLD) - 1
g = pairs[pairs.g.between(-0.30, 0.50)].groupby("id").g.median().rename("ebitda_g")

co = (per_co.rename(columns={"da_pct": "da", "capex_nwc_pct": "capex_nwc"})
            .join(g, how="inner").dropna())
co.to_csv("data/brightquery_company_params.csv")

print(len(co), "companies")
print(co.describe().round(3))
print(co.corr(method="spearman").round(2))   # real correlations between the inputs

773 companies
            da  capex_nwc  ebitda_g
count  773.000    773.000   773.000
mean     0.387      0.456     0.055
std      0.216      0.385     0.087
min      0.001     -0.494    -0.230
25%      0.201      0.196     0.006
50%      0.374      0.444     0.043
75%      0.565      0.694     0.088
max      0.795      1.500     0.411
             da  capex_nwc  ebitda_g
da         1.00       0.51     -0.06
capex_nwc  0.51       1.00      0.04
ebitda_g  -0.06       0.04      1.00


In [12]:
# How many companies did each filtering step remove?
per_co_all = pos.groupby("id")[["da_pct", "capex_nwc_pct"]].median()   # before outlier filter
n_total = len(per_co_all)
n_missing = per_co_all.isna().any(axis=1).sum()                          # not enough data to calculate
n_outliers = (n_total - n_missing) - len(per_co)                         # removed by our bounds

print(f"Companies with positive EBITDA:     {n_total}")
print(f"  dropped, insufficient data:        {n_missing}")
print(f"  dropped, outside outlier bounds:   {n_outliers}  ({n_outliers / (n_total - n_missing):.1%})")
print(f"  kept:                              {len(per_co)}")

# Same check for EBITDA growth (counted in 5-year windows, not companies)
all_growth = (pairs.ebitda_fwd / pairs.ebitda) ** (1 / HOLD) - 1
print(f"\nGrowth windows: {len(all_growth)}, dropped as outliers: {len(all_growth) - len(growth)} "
      f"({1 - len(growth) / len(all_growth):.1%})")

Companies with positive EBITDA:     5031
  dropped, insufficient data:        271
  dropped, outside outlier bounds:   3950  (83.0%)
  kept:                              810

Growth windows: 41793, dropped as outliers: 1945 (4.7%)
